Import and setup

In [1]:
# config
from pathlib import Path
import sys
import datetime as Date
import json

# Nếu notebook nằm trong thư mục notebooks/
PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source exists:", SRC_DIR.exists())
print("Pipeline exists:", (SRC_DIR / "pipeline").is_dir())

from pipeline.logger import setup_logger, log_execution
from pipeline.reader import read_data, create_content_hash

from pipeline.database import create_db_engine
from pipeline.executor import execute_sql, truncate_table, query_sql

# database
DRY_RUN = True
ROLLBACK = True
engine = create_db_engine()

# table init
execute_sql(
    engine,
    """
    CREATE SCHEMA IF NOT EXISTS bronze;
    CREATE TABLE IF NOT EXISTS bronze.ingestion (
        id SERIAL PRIMARY KEY,
        source_name VARCHAR(255) NOT NULL,
        content_hash VARCHAR(64) NOT NULL,
        loaded_time TIMESTAMP NOT NULL,
        metadata JSONB
    );
    """,
)

Project root: d:\_HCMUT\_Nam4\k261\CO4031-Datawarehouse\Global-Superstore-warehouse-and-DSS
Source exists: True
Pipeline exists: True


{'success': True, 'rowcount': -1, 'rolled_back': False}

Load file

In [3]:
# load file
logger = setup_logger(name="bronze",run_id="bronze_001")
source_name = "Global_Superstore.xls"
# source_nname = "Sample-Superstore.csv"

@log_execution(logger)
def run_extract(file_path):
    return read_data(file_path)

df = run_extract(PROJECT_ROOT / "data/raw" / source_name)["data"]
content_hash = create_content_hash(PROJECT_ROOT / "data/raw" / source_name)
loaded_time = Date.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Data loaded at: {loaded_time}")
print(f"Content hash: {content_hash}")

print(df.head(5))
print(df.shape)

2026-11-03 01:37:23,201 | INFO | run_id=bronze_001 | Started: run_extract
2026-11-03 01:37:26,537 | INFO | run_id=bronze_001 | Succeeded: run_extract
2026-11-03 01:37:26,547 | INFO | run_id=bronze_001 | Finished: run_extract | duration=3.35s


Data loaded at: 2026-11-03 01:37:26
Content hash: 5cc6581be17e438a88f0c15edb8d65f17a87ffd7dc043f20b1eca0dc72d37d77
   Row ID        Order ID Order Date  Ship Date       Ship Mode Customer ID  \
0       1  MX-2014-143658 2014-10-02 2014-10-06  Standard Class    SC-20575   
1       2  MX-2012-155047 2012-10-15 2012-10-20  Standard Class    KW-16570   
2       3  MX-2012-155047 2012-10-15 2012-10-20  Standard Class    KW-16570   
3       4  MX-2012-155047 2012-10-15 2012-10-20  Standard Class    KW-16570   
4       5  MX-2012-155047 2012-10-15 2012-10-20  Standard Class    KW-16570   

    Customer Name   Segment           City             State  ...  \
0    Sonia Cooley  Consumer    Mexico City  Distrito Federal  ...   
1  Kelly Williams  Consumer  Dos Quebradas         Risaralda  ...   
2  Kelly Williams  Consumer  Dos Quebradas         Risaralda  ...   
3  Kelly Williams  Consumer  Dos Quebradas         Risaralda  ...   
4  Kelly Williams  Consumer  Dos Quebradas         Risaralda  ...

Check if file processed? Do not duplicate

In [7]:
already_processed = query_sql(
    engine,
    """
    SELECT EXISTS (
        SELECT 1
        FROM bronze.ingestion
        WHERE content_hash = :content_hash
    )
    """,
    {"content_hash": content_hash}
)
print(f"Already processed: {already_processed}")

if already_processed["exists"][0]:  # Check if the first row and first column is True
    logger.info(
        "Skipping ingestion: content hash %s is already recorded.",
        content_hash,
    )
else:
    logger.info("Content hash not found; ingestion can proceed.")


2026-11-03 01:38:50,336 | INFO | run_id=bronze_001 | Content hash not found; ingestion can proceed.


Already processed:    exists
0   False


Check enough required column

In [17]:
REQUIRED_COLUMNS = [
    "Row ID",
    "Order ID",
    "Order Date",
    "Ship Date",
    "Ship Mode",
    "Customer ID",
    "Customer Name",
    "Segment",
    "Country",
    "City",
    "State",
    "Region",
    "Product ID",
    "Category",
    "Sub-Category",
    "Product Name",
    "Sales",
    "Quantity",
    "Discount",
    "Profit"
]

OPTIONAL_COLUMNS = [
    "Postal Code",
    "Market",
    "Shipping Cost",
    "Order Priority"
]

ALL_COLUMNS = REQUIRED_COLUMNS + OPTIONAL_COLUMNS

In [18]:
checked_df = df.copy()

missing = [
    col for col in REQUIRED_COLUMNS
    if col not in checked_df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

for col in OPTIONAL_COLUMNS:
    if col not in df.columns:
        checked_df[col] = None

print(checked_df.shape)

(51290, 24)


Extract market, postal table

In [19]:
if {"Country", "Market"}.issubset(checked_df.columns):
    # get all unique set of Country -> Market
    unique_market = checked_df[["Country", "Market"]].drop_duplicates()
    print(unique_market)
    market_count = unique_market.shape[0]

if {"City", "State", "Country", "Postal Code"}.issubset(checked_df.columns):
    # only get the rows that have Postal Code
    have_pin = checked_df[checked_df["Postal Code"].notnull()]
    # get all unique set of  City, State, Country -> Postal Code 
    unique_pin = have_pin[["City", "State", "Country", "Postal Code"]].drop_duplicates()
    print(unique_pin)
    pin_count = unique_pin.shape[0]

                 Country  Market
0                 Mexico   LATAM
1               Colombia   LATAM
6                 Brazil   LATAM
11             Nicaragua   LATAM
16                Panama   LATAM
...                  ...     ...
46325           Ethiopia  Africa
46426  Equatorial Guinea  Africa
47561            Bahrain    EMEA
48507               Chad  Africa
49764            Eritrea  Africa

[149 rows x 2 columns]
                  City           State        Country  Postal Code
31296        Henderson        Kentucky  United States      42420.0
31298      Los Angeles      California  United States      90036.0
31299  Fort Lauderdale         Florida  United States      33311.0
31301      Los Angeles      California  United States      90032.0
31308          Concord  North Carolina  United States      28027.0
...                ...             ...            ...          ...
41056       Springdale        Arkansas  United States      72762.0
41097             Lodi      California  Unit

Generate ingestion metadata and insert ingestion

In [20]:
metadata = {
    "market_count": market_count,
    "pin_count": pin_count,
    "rows": checked_df.shape[0],
}
print(metadata)

execute_sql(
    engine,
    """
    INSERT INTO bronze.ingestion (source_name, content_hash, loaded_time, metadata) VALUES (:source_name, :content_hash, :loaded_time, :metadata)
    """,
    {
        "source_name": source_name,
        "content_hash": content_hash,
        "loaded_time": loaded_time,
        "metadata": json.dumps(metadata)
    }
)

{'market_count': 149, 'pin_count': 632, 'rows': 51290}


1

check if database available? table? exist

insert raw_orders -> amount of column?

insert ingestion table

In [22]:
# restart cell here =))
truncate_table(engine, "bronze", "ingestion")

-1